# LLaMA-350M Training with COSMOS Optimizer
**Paper:** *COSMOS: A Hybrid Adaptive Optimizer for Efficient Training of Large Language Models* (ICLR 2026 submission)

COSMOS decomposes each gradient matrix into a **leading eigensubspace** (handled with a SOAP-like adaptive update) and a **residual eigensubspace** (handled with a MUON-like Newton-Schulz preconditioner), achieving near-SOAP accuracy at near-MUON memory cost.

---
### Notebook Structure
1. Install dependencies  
2. COSMOS optimizer implementation  
3. Build LLaMA-350M model  
4. Build optimizer (COSMOS for hidden layers, AdamW for embeddings)  
5. Build C4 dataloader  
6. Training loop  
7. Run training

## 1. Install Dependencies

In [1]:
# Install required packages
# (skip any already present in your environment)
%pip install torch transformers datasets tokenizers accelerate -q

## 2. Imports & Logging

In [17]:
import os
import math
import logging
import platform
import subprocess
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    LlamaConfig,
    LlamaForCausalLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from datasets import load_dataset
import wandb

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("train")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

INFO:train:Using device: cuda


## 3. COSMOS Optimizer
### 3a. Newton-Schulz Zero-Power Preconditioner
Approximates the matrix sign operator (i.e., `UV^T` from SVD) via 5 iterations — used on the residual subspace.

In [18]:
@torch.compile
def zeropower_via_newtonschulz5(G, steps=10, eps=1e-7):
    """
    Approximate matrix sign operator via Newton-Schulz iterations.
    Used by COSMOS on the residual (non-leading) eigensubspace — the MUON component.
    """
    assert len(G.shape) == 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.bfloat16()
    X /= (X.norm() + eps)          # ensure top singular value <= 1
    if G.size(0) > G.size(1):
        X = X.T
    for _ in range(steps):
        A = X @ X.T
        B = A @ X
        X = a * X + b * B + c * A @ B
    if G.size(0) > G.size(1):
        X = X.T
    return X

### 3b. COSMOS Functional Core
For each 2-D weight matrix, COSMOS:
1. **Tracks the leading eigensubspace** `U` of the gradient second-moment via a one-step power iteration + QR.
2. **Applies an Adam/SOAP-like adaptive update** (`A_t`) in that low-rank subspace.
3. **Applies a MUON-like Newton-Schulz update** (`B_t`) on the orthogonal complement.
4. **Combines** the two components: `Ã_t = A_t + γ · B_t`.

1-D parameters and very large matrices (>10 000 rows) fall back to standard AdamW.

In [19]:
wandb.init(
    project="cosmos-llama-350m",
    name="llama350m_cosmos_run1",
    config={
        "hidden_size": 1024,
        "num_layers": 16,
        "num_heads": 16,
        "seq_len": 1024,
        "cosmos_lr": 5e-4,
        "adam_lr": 2e-3,
        "rank": 64,
        "beta1": 0.9,
        "beta2": 0.98,
        "max_steps": 5000,
        "warmup_steps": 500,
        "batch_size": 2,
    }
)

def get_gpu_info():
    info = {}
    if torch.cuda.is_available():
        info["num_gpus"] = torch.cuda.device_count()
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["cuda_version"] = torch.version.cuda
        try:
            smi = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total",
                 "--format=csv,noheader"]
            ).decode().strip()
            info["nvidia_smi"] = smi
        except:
            info["nvidia_smi"] = "unavailable"
    return info

wandb.config.update({
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    **get_gpu_info()
})

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sethikrish221 (sethikrish221-university-of-waterloo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [20]:
def cosmos(
    params: List[Tensor],
    grads: List[Tensor],
    exp_avgs: List[Tensor],
    exp_avg_sqs: List[Tensor],
    exp_avgs_GG: List[Tensor],
    exp_avgs_P: List[Tensor],
    max_exp_avg_sqs: List[Tensor],
    state_steps: List[int],
    *,
    amsgrad: bool,
    beta1: float,
    beta2: float,
    lr: float,
    weight_decay: float,
    eps: float,
    maximize: bool,
    ratio: float,
    gamma: float,
    nestrov: bool,
):
    for i, param in enumerate(params):
        grad        = grads[i] if not maximize else -grads[i]
        exp_avg     = exp_avgs[i]
        exp_avg_sq  = exp_avg_sqs[i]
        exp_avg_gg  = exp_avgs_GG[i]
        exp_avg_p   = exp_avgs_P[i]
        step        = state_steps[i]

        bias_correction1 = (1 - beta1 ** step) / (1 - beta1)
        bias_correction2 = 1 - beta2 ** step

        # ── COSMOS update for 2-D hidden-layer weights ────────────────────
        if len(param.size()) == 2 and param.size(0) <= 10000:
            exp_avg.mul_(beta1).add_(grad)

            if step == 1:
                # Initialise leading eigensubspace via full SVD on first step
                W = torch.matmul(grad.T, grad)
                U, _, _ = torch.linalg.svd(W, full_matrices=False)
                exp_avg_p.data = U[:, :exp_avg_gg.size(0)]
                exp_avg_gg = torch.matmul(
                    torch.matmul(exp_avg_p.T, grad.T),
                    torch.matmul(grad, exp_avg_p)
                ) * (1 - beta2)
            else:
                # One-step power iteration to track the leading eigensubspace
                t = exp_avg_p.detach().clone().T
                exp_avg_p = (
                    beta2 * torch.matmul(exp_avg_p, exp_avg_gg)
                    + (1 - beta2) * torch.matmul(grad.T, torch.matmul(grad, exp_avg_p))
                )
                exp_avg_p, _ = torch.linalg.qr(exp_avg_p, mode='reduced')
                t = torch.matmul(t, exp_avg_p)
                exp_avg_gg = (
                    beta2 * torch.matmul(t.T, torch.matmul(exp_avg_gg, t))
                    + (1 - beta2) * torch.matmul(
                        torch.matmul(grad, exp_avg_p).T,
                        torch.matmul(grad, exp_avg_p)
                    )
                )

            scale        = (grad.size(0) * grad.size(1)) ** 0.5
            low_rank_grad = torch.matmul(grad, exp_avg_p)
            exp_avg_sq.mul_(beta2).addcmul_(
                low_rank_grad, low_rank_grad.conj(), value=1 - beta2
            )

            # Momentum correction (Nesterov optional)
            if nestrov:
                grad.add_(exp_avg, alpha=beta1)
                grad.mul_(1 / (1 + beta1 * bias_correction1))
            else:
                grad.mul_(1 / bias_correction1)

            # ── SOAP component: adaptive update in leading subspace (A_t) ──
            t  = torch.matmul(grad, exp_avg_p)
            t1 = t / ((exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps))
            t1 = torch.matmul(t1, exp_avg_p.T)

            # ── MUON component: Newton-Schulz on residual subspace (B_t) ──
            t   = grad - torch.matmul(t, exp_avg_p.T)
            t   = zeropower_via_newtonschulz5(t, steps=5)
            t   = scale * t / (t.norm() + eps)

            # ── Combine: G̃_t = A_t + γ · B_t ──────────────────────────────
            t1.add_(t, alpha=gamma)

            if weight_decay > 0:
                param.data.mul_(1 - lr * weight_decay)

            param.add_(t1 / (t1.norm() + eps), alpha=-scale * ratio * lr)

            exp_avgs_P[i].copy_(exp_avg_p)
            exp_avgs_GG[i].copy_(exp_avg_gg)

        # ── AdamW fallback for embeddings / 1-D params / very large matrices
        else:
            bias_correction1 = 1 - 0.9 ** step
            exp_avg.mul_(beta1).add_(grad, alpha=0.1)
            exp_avg_sq.mul_(beta2).addcmul_(grad, grad.conj(), value=1 - beta2)

            if amsgrad:
                torch.maximum(max_exp_avg_sqs[i], exp_avg_sq, out=max_exp_avg_sqs[i])
                denom = (max_exp_avg_sqs[i].sqrt() / math.sqrt(bias_correction2)).add_(eps)
            else:
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)

            step_size = lr / bias_correction1
            if weight_decay > 0:
                param.data.mul_(1 - lr * weight_decay)
            param.addcdiv_(exp_avg, denom, value=-step_size)

### 3c. COSMOS Optimizer Class

In [21]:
class COSMOS(Optimizer):
    """
    COSMOS: Hybrid SOAP + MUON optimizer.

    Args:
        params       : model parameters
        lr           : learning rate for hidden layers
        betas        : (β1, β2) momentum coefficients  [paper: (0.9, 0.98)]
        eps          : numerical stability term
        lr_ratio     : scale factor applied to the final update norm
        rank         : rank r of the leading eigensubspace  [paper: 64]
        weight_decay : L2 regularisation coefficient
        gamma        : weight of the MUON component  [paper heuristic: lr/adam_lr]
        nestrov      : use Nesterov-style momentum
        amsgrad      : use AMSGrad variant
        maximize     : maximise the objective (default: False)
    """

    def __init__(
        self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
        lr_ratio=0.1, rank=64, weight_decay=0, gamma=0.2,
        nestrov=True, amsgrad=False, *, maximize: bool = False,
    ):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta[0]: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta[1]: {betas[1]}")
        if not 0.0 <= weight_decay:
            raise ValueError(f"Invalid weight_decay: {weight_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps,
                        weight_decay=weight_decay, amsgrad=amsgrad, maximize=maximize)
        super().__init__(params, defaults)
        self.lr_ratio = lr_ratio
        self.rank     = rank
        self.nestrov  = nestrov
        self.gamma    = gamma

    def __setstate__(self, state):
        super().__setstate__(state)
        for group in self.param_groups:
            group.setdefault('amsgrad', False)
            group.setdefault('maximize', False)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            params_with_grad, grads             = [], []
            exp_avgs, exp_avg_sqs               = [], []
            exp_avgs_GG, exp_avgs_P             = [], []
            max_exp_avg_sqs, state_steps        = [], []
            beta1, beta2 = group['betas']

            for p in group['params']:
                if p.grad is None:
                    continue
                if p.grad.is_sparse:
                    raise RuntimeError('COSMOS does not support sparse gradients')

                params_with_grad.append(p)
                grads.append(p.grad)

                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)

                    if len(p.size()) == 2 and p.size(0) <= 10000:
                        state['exp_avg_GG'] = torch.zeros(
                            self.rank, self.rank, dtype=p.dtype, device=p.device)
                        state['exp_avg_P']  = torch.zeros(
                            p.size(1), self.rank, dtype=p.dtype, device=p.device)
                        state['exp_avg_sq'] = torch.zeros(
                            p.size(0), self.rank, dtype=p.dtype, device=p.device)
                    else:
                        state['exp_avg_GG'] = torch.zeros(0)
                        state['exp_avg_P']  = torch.zeros(0)
                        state['exp_avg_sq'] = torch.zeros_like(p)

                    if group['amsgrad']:
                        state['max_exp_avg_sq'] = torch.zeros_like(p)

                exp_avgs.append(state['exp_avg'])
                exp_avg_sqs.append(state['exp_avg_sq'])
                exp_avgs_GG.append(state['exp_avg_GG'])
                exp_avgs_P.append(state['exp_avg_P'])

                if group['amsgrad']:
                    max_exp_avg_sqs.append(state['max_exp_avg_sq'])

                state['step'] += 1
                state_steps.append(state['step'])

            cosmos(
                params_with_grad, grads,
                exp_avgs, exp_avg_sqs, exp_avgs_GG, exp_avgs_P,
                max_exp_avg_sqs, state_steps,
                amsgrad=group['amsgrad'],
                beta1=beta1, beta2=beta2,
                lr=group['lr'],
                weight_decay=group['weight_decay'],
                eps=group['eps'],
                maximize=group['maximize'],
                ratio=self.lr_ratio,
                gamma=self.gamma,
                nestrov=self.nestrov,
            )
        return loss

print("✅  COSMOS optimizer defined.")

✅  COSMOS optimizer defined.


## 4. Build LLaMA-350M Model
Architecture per the paper (Appendix A):

| Hyperparameter | Value |
|---|---|
| Hidden size | 1024 |
| Layers | 16 |
| Attention heads | 16 (head-dim = 64) |
| MLP hidden | ≈ 8/3 × 1024 ≈ 2752 |
| Activation | SiLU |
| Position encoding | RoPE |
| Sequence length | 1024 |

In [22]:
def build_llama_350m() -> LlamaForCausalLM:
    config = LlamaConfig(
        hidden_size=1024,
        intermediate_size=2752,       # ≈ (8/3) * 1024, rounded to nearest 64
        num_hidden_layers=16,
        num_attention_heads=16,
        max_position_embeddings=1024,
        rms_norm_eps=1e-5,
        vocab_size=32000,             # T5 tokenizer vocab size
        hidden_act="silu",
        rope_theta=10000.0,
    )
    model = LlamaForCausalLM(config)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f"Model built — {n_params:.1f}M parameters")
    return model

model = build_llama_350m().to(device)

INFO:train:Model built — 267.9M parameters


## 5. Build Optimizer
Following the paper's setup:
- **Hidden layers** → COSMOS (`cosmos_lr = 5e-4`)
- **Embedding + lm_head** → AdamW (`adam_lr = 2e-3`)
- **γ heuristic**: `γ = cosmos_lr / adam_lr` — paper shows this reliably lands in the optimal 0.25–0.5 range without extra tuning

In [23]:
# ── Hyperparameters (match paper Table 2 / Appendix A.1.2) ──────────────────
COSMOS_LR    = 5e-4      # searched over {1e-4, 2e-4, 5e-4, 1e-3, 2e-3}
ADAM_LR      = 2e-3      # fixed for embedding + output layers
RANK         = 64        # low-rank subspace dimension r
WEIGHT_DECAY = 0.0       # paper sets weight_decay = 0
BETA1, BETA2 = 0.9, 0.98 # β2=0.98 per Liu et al. 2019 / Appendix A

# γ = cosmos_lr / adam_lr  →  paper heuristic to avoid tuning γ separately
GAMMA = COSMOS_LR / ADAM_LR
logger.info(f"γ = {GAMMA:.4f}  (target range 0.25 – 0.5)")

def build_optimizer(model):
    embed_params, cosmos_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "embed_tokens" in name or "lm_head" in name:
            embed_params.append(param)
        else:
            cosmos_params.append(param)

    logger.info(
        f"COSMOS params: {sum(p.numel() for p in cosmos_params)/1e6:.1f}M  |  "
        f"AdamW params:  {sum(p.numel() for p in embed_params)/1e6:.1f}M"
    )

    cosmos_opt = COSMOS(
        cosmos_params,
        lr=COSMOS_LR,
        betas=(BETA1, BETA2),
        eps=1e-8,
        rank=RANK,
        gamma=GAMMA,
        weight_decay=WEIGHT_DECAY,
        nestrov=True,
    )
    adam_opt = torch.optim.AdamW(
        embed_params,
        lr=ADAM_LR,
        betas=(BETA1, BETA2),
        eps=1e-8,
        weight_decay=WEIGHT_DECAY,
    )
    return cosmos_opt, adam_opt

cosmos_opt, adam_opt = build_optimizer(model)
print("✅  Optimizers ready.")

INFO:train:γ = 0.2500  (target range 0.25 – 0.5)
INFO:train:COSMOS params: 202.4M  |  AdamW params:  65.5M


✅  Optimizers ready.


## 6. Schedulers
Linear warm-up for the first 10 % of steps, then linear decay to 0 — matching the paper's default schedule.

In [24]:
MAX_STEPS    = 5000     # paper: 5000 steps for 350M model
WARMUP_STEPS = int(0.10 * MAX_STEPS)   # 10 % warmup

cosmos_sched = get_linear_schedule_with_warmup(
    cosmos_opt,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=MAX_STEPS,
)
adam_sched = get_linear_schedule_with_warmup(
    adam_opt,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=MAX_STEPS,
)
print(f"✅  Schedulers ready — {MAX_STEPS} total steps, {WARMUP_STEPS} warmup steps.")

✅  Schedulers ready — 5000 total steps, 500 warmup steps.


In [25]:
def get_gpu_info():
    info = {}
    if torch.cuda.is_available():
        info["num_gpus"] = torch.cuda.device_count()
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["cuda_version"] = torch.version.cuda

        try:
            smi = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name,memory.total",
                 "--format=csv,noheader"]
            ).decode()
            info["nvidia_smi"] = smi
        except:
            pass
    return info

wandb.config.update({
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    **get_gpu_info()
})

## 7. Dataloader
Streams the **C4 (en)** dataset tokenised with the **T5 tokeniser**, packed to `seq_len = 1024` — identical to the paper's setup.

> **Tip:** Set `SUBSET_SIZE` to a small integer (e.g. `2000`) for a quick smoke-test before launching the full run.

In [26]:
SEQ_LEN     = 1024
BATCH_SIZE  = 2          # increase if VRAM allows; paper uses effective ~2000 tokens/batch
NUM_WORKERS = 2
SUBSET_SIZE = None       # set e.g. 2000 for a smoke-test; None = full C4

tokenizer = AutoTokenizer.from_pretrained("t5-base")
tokenizer.pad_token = tokenizer.eos_token

class TokenDataset(torch.utils.data.IterableDataset):
    """Wraps a HuggingFace streaming dataset, tokenising on the fly."""
    def __init__(self, hf_dataset, tok, length):
        self.ds, self.tok, self.len = hf_dataset, tok, length

    def __iter__(self):
        for sample in self.ds:
            enc = self.tok(
                sample["text"],
                truncation=True,
                max_length=self.len,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].squeeze(0)
            yield {"input_ids": input_ids, "labels": input_ids.clone()}

raw_dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)  # ← fixed
if SUBSET_SIZE is not None:
    raw_dataset = raw_dataset.take(SUBSET_SIZE)

dataset = TokenDataset(raw_dataset, tokenizer, SEQ_LEN)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                     num_workers=NUM_WORKERS, pin_memory=True)
print("✅  Dataloader ready.")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

✅  Dataloader ready.


## 8. Training Loop

In [27]:
scaler = GradScaler()
global_step = 0
SAVE_EVERY = 500
SAVE_DIR = "./llama350m_cosmos"
os.makedirs(SAVE_DIR, exist_ok=True)

model.train()

for batch in loader:
    if global_step >= MAX_STEPS:
        break

    batch = {k: v.to(device) for k, v in batch.items()}

    with autocast():
        outputs = model(**batch)
        loss = outputs.loss

    scaler.scale(loss).backward()

    scaler.step(cosmos_opt)
    scaler.step(adam_opt)
    scaler.update()

    cosmos_opt.zero_grad()
    adam_opt.zero_grad()

    cosmos_sched.step()
    adam_sched.step()

    global_step += 1

    wandb.log({
        "loss": loss.item(),
        "lr_cosmos": cosmos_sched.get_last_lr()[0],
        "lr_adam": adam_sched.get_last_lr()[0],
        "step": global_step,
        "gpu_memory_allocated_mb": torch.cuda.memory_allocated() / 1e6 if torch.cuda.is_available() else 0,
    })

    if global_step % 50 == 0:
        logger.info(f"Step {global_step}/{MAX_STEPS} — Loss: {loss.item():.4f}")

    if global_step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(SAVE_DIR, f"ckpt_step{global_step}.pt")
        torch.save(model.state_dict(), ckpt_path)
        logger.info(f"Checkpoint saved → {ckpt_path}")

Streaming output truncated to the last 5000 lines.
  with autocast():
/tmp/ipykernel_1494/2836027020.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1494/2836027020.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1494/2836027020.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1494/2836027020.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1494/2836027020.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_1494/2836027020

KeyboardInterrupt: 